### RAG Pipeline --Data Ingestion to Vector DB Pipeline


In [3]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from pathlib import Path

#Read all the pdf inside directory and load them into a list of documents:
def process_all_pdfs(pdf_directory):
    """Process all the pdf files in the directory"""
    pdf_documents = []
    pdf_dir = Path(pdf_directory)

    #find all pdf files recursively:
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #add source information to meta data:
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            pdf_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {pdf_file.name}")
        
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(pdf_documents)}")
    return pdf_documents

#process all pdf files in the directory:
all_pdf_documents = process_all_pdfs("../pdf")


         

    

Found 2 PDF files to process

Processing:The-Complete-Guide-to-Building-Skill-for-Claude.pdf
Loaded 33 documents from The-Complete-Guide-to-Building-Skill-for-Claude.pdf

Processing:Md Shamserul Haque CV MLOps.pdf
Loaded 2 documents from Md Shamserul Haque CV MLOps.pdf

Total documents loaded: 35


In [6]:
#to check all the loaded documents:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-26T14:22:24-06:00', 'moddate': '2026-01-26T14:22:28-06:00', 'trapped': '/False', 'source': '../pdf/The-Complete-Guide-to-Building-Skill-for-Claude.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1', 'source_file': 'The-Complete-Guide-to-Building-Skill-for-Claude.pdf', 'file_type': 'pdf'}, page_content='The Complete Guide \nto Building Skills \nfor Claude'),
 Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-26T14:22:24-06:00', 'moddate': '2026-01-26T14:22:28-06:00', 'trapped': '/False', 'source': '../pdf/The-Complete-Guide-to-Building-Skill-for-Claude.pdf', 'total_pages': 33, 'page': 1, 'page_label': '2', 'source_file': 'The-Complete-Guide-to-Building-Skill-for-Claude.pdf', 'file_type': 'pdf'}, page_content='Contents\nIntroduction 3\nFundamentals 4\nPlanning and design 7\nT estin

In [7]:
#Text splitting into chunks:

def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_documents = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_documents)} chunks")

    #show example of chunks:
    if split_documents:
        print(f"\nExample chunk:")
        print(f"Content:{split_documents[0].page_content[:500]}...")  # Show first 500 characters
        print(f"Metadata:{split_documents[0].metadata}")

    return split_documents

In [9]:
#to see the chunks of the documents:
chunks = split_documents_into_chunks(all_pdf_documents)
chunks

Split 35 documents into 65 chunks

Example chunk:
Content:The Complete Guide 
to Building Skills 
for Claude...
Metadata:{'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-26T14:22:24-06:00', 'moddate': '2026-01-26T14:22:28-06:00', 'trapped': '/False', 'source': '../pdf/The-Complete-Guide-to-Building-Skill-for-Claude.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1', 'source_file': 'The-Complete-Guide-to-Building-Skill-for-Claude.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-26T14:22:24-06:00', 'moddate': '2026-01-26T14:22:28-06:00', 'trapped': '/False', 'source': '../pdf/The-Complete-Guide-to-Building-Skill-for-Claude.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1', 'source_file': 'The-Complete-Guide-to-Building-Skill-for-Claude.pdf', 'file_type': 'pdf'}, page_content='The Complete Guide \nto Building Skills \nfor Claude'),
 Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-26T14:22:24-06:00', 'moddate': '2026-01-26T14:22:28-06:00', 'trapped': '/False', 'source': '../pdf/The-Complete-Guide-to-Building-Skill-for-Claude.pdf', 'total_pages': 33, 'page': 1, 'page_label': '2', 'source_file': 'The-Complete-Guide-to-Building-Skill-for-Claude.pdf', 'file_type': 'pdf'}, page_content='Contents\nIntroduction 3\nFundamentals 4\nPlanning and design 7\nT estin

### Embedding and Vector Store DB:

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer 
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/md_haque@optum.com/Library/CloudStorage/OneDrive-UHG/Desktop/agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#Embedding:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name (str): Hugging Face model name for sentence transformers.
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts (List[str]): List of text strings to embed.

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim).
        """
        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
#Initialize the embedding manager:
embedding_manager = EmbeddingManager()
embedding_manager

        
            
 

### Vector Store:

In [ ]:
class VectorStore:
    """Manages document embeddings in ChromaDB  vector store"""

    def __init__(self,collection_name:str = "pdf_documents",persist_directory: str = "../RAG/vector_store"):
        """
        Initialize the vector store.

        Args:
            collection_name (str): Name of the ChromaDB collection to use.
            persist_directory (str): Directory to persist the ChromaDB data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        """
        Initialize the ChromaDB client and collection.
        """
        try:
            #create persistent ChromaDB client:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create the collection:
            self.collection = self.client.get_or_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings for RAG."}
            )

            print(f"ChromaDB initialized. Collection {self.collection_name}")
            print(f"Current number of documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the ChromaDB collection.

        Args:
            documents (List[Any]): List of document objects (with metadata).
            embeddings (np.ndarray): Corresponding embeddings for the documents.
        """
        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to the vector store...")

       # Prepare data for ChromaDB:
        ids = []  # Generate unique IDs for each document
        metadatas = []  # Extract metadata from documents
        documents_texts = []  # Extract text content from documents
        embeddings_list = [] # Convert embeddings to list format

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):

            #Generate a unique ID for each document:
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" 
            ids.append(doc_id)

            #Prepare metadata:
            metadata = dict(doc.metadata)  # Ensure metadata is a dictionary
            metadata['doc_index'] = i  # Add index for reference
            metadata['continue'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document text content:
            documents_texts.append(doc.page_content)

            #Embedding:
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list

        #Add to ChromaDB collection:
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Current number of documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to ChromaDB: {e}")
            raise

vectorstore = VectorStore()
vectorstore

            

In [ ]:
chunks

In [ ]:
#Convert the text to embeddings:
texts = [doc.page_content for doc in chunks]

#Generate embeddings for the chunks:
embeddings = embedding_manager.generate_embeddings(texts)

#Stote the chunks and their embeddings in the vector store:
vectorstore.add_documents(chunks, embeddings)

### Retriever Pipeline from Vector Store:

In [ ]:
class RAGRetriever:
    """Retrieves relevant documents from the vector store based on a query."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAG retriever.

        Args:
            vector_store (VectorStore): Instance of the VectorStore class.
            embedding_manager (EmbeddingManager): Instance of the EmbeddingManager class.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents based on the query.

        Args:
            query (str): The input query string.
            top_k (int): Number of top relevant documents to retrieve.
            score_threshold (float): Minimum similarity score threshold for retrieved documents.

        Returns:
            List of dictionaries containing retrieved documents and their metadata.
        """
        #Generate embedding for the query:
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store for relevant documents:
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #Process the results to filter based on score_threshold:
            retrieved_docs = []

            if results['documents']  and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, metadata, distance) in enumerate(zip(documents, metadatas, distances, ids)):

                    #Convert distance to similarity score (ChromaDB returns distance, we want similarity):
                    similarity_score = 1 - distance  # Convert distance to similarity

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": documents,
                            "metadata": metadata,
                            "similarity": similarity_score,
                            "distance": distance,
                            "rank": i + 1  # Rank starts from 1

                        })
                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}.")
            else:
                print("No documents found above the score threshold.")
                
                return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
                   
                     

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("What is the purpose of this document?", top_k=5, score_threshold=0.5)

In [ ]:
rag_retriever.retrieve("Unified Multi-task Learning Framework ?")

### Integration of vector DB context pipeline with LLM output:

In [ ]:
#Simple RAG pipeline with Groq LLM:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#Initialize the Groq LLM(set environment variable GROQ_API_KEY with your Groq API key):
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key= groq_api_key, model_name="gemma2-9b-it",temperature=0.1,tokens=1024)

#simple RAG function: retrieve context + generate response:
def rag_simple(query, retriever, llm , top_k = 3):
    #Retrieve the context from the vector store:
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else""
    if not context:
        return "No relevant information found in the documents."
    
    #generate the answer using the groq LLM:
    prompt = f"""Use the following context to answer the question concisely.

     
        context:
        {context}
        Question: {query}
        Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

        

In [ ]:
answer=rag_simple("What is attention mechanism?", rag_retriever, llm)
print("Answer:", answer)

### Enhanced RAG Pipeline Features:

In [ ]:
#Enhanced RAG pipeline:
from pydoc import doc


def rag_advanced(query, retriever, llm, top_k=5, min_score = 0.2, return_context=False):
    #Retrieve the context from the vector store:
    """
    RAG pipline with extra features;
     - Returns answers, sources, confidence scores, and optionally the context used for generating the answer.

    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
            "answer": "No relevant information found in the documents above the score threshold.",
            "sources": [],
            "confidence_scores": 0.0,
            "context": ""
        }
    
    #Prepare the context and sources for the answer generation:
    context = "\n\n".join([doc['content'] for doc in results]) 
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'Unknown')),
        'page': doc['metadata'].get('page', 'Unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'  # First 300 characters of the content
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    #Generate answer:
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\nQuestion: {query}\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence_scores': confidence
    }
    if return_context:
        output['context'] = context
    return output

#example usage of the enhanced RAG pipeline:
result = rag_advanced("What is attention mechanism?", rag_retriever, llm,top_k= 3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence Score:", result['confidence_scores'])
print("Context Preview:", result['context'][:300] + '...')  # Show first 300 characters of the context

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])